In [1]:
import gc, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from google.colab import drive, runtime

IN_COLAB = True
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Colab Notebooks/IV Project-Apr 3, 2026')
sys.path.insert(0, str(ROOT))

from src.paths import CLEAN_DATA_V2, OUTPUT
from src.benchmark import analytic_benchmark
from src.helper import make_run_dir, _notebook_stem
from src.lstm import LSTMModel
from src.metrics import metrics, gain
from src.model3_utils import (
    FEATURE_SETS, TARGET, LOOKBACK,
    precompute_split_structure, build_sequences_from_cache,
    scale_sequences, SequenceDataset,
    detect_device, compute_batch_size,
    train_seq_model, predict_seq,
    hw_predict_aligned, save_seq_run,
    print_config, print_feature_set_summary,
)

Mounted at /content/drive


- Runtime Recommendations

1. PATIENCE: 25 → 10-12  (cuts wasted no-improvement epochs)
2. LR_PATIENCE: 8 → 5    (LR drops faster → faster convergence)
3. LOOKBACK: 20 → 10     (halves sequence tensor size; less GPU memory)
   Change in src/model3_utils.py: LOOKBACK = 10
4. MAX_EPOCHS: 100 → 50  (FC models converged well under 100 epochs)

## Configuration

In [2]:
DATA_SETS = ['chro_B', 'chro_C', 'chro_D']
NOTEBOOK_NAME = _notebook_stem()

SEED = 42
MAX_EPOCHS = 100
PATIENCE = 12       # 25
LR_PATIENCE = 5     # 8
LR_FACTOR = 0.3
WARMUP_EPOCHS = 5
HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.1
BASE_LR = 1e-3
BASE_BATCH = 512

# Override feature sets locally if needed:
FEATURE_SETS = {k: v for k, v in FEATURE_SETS.items() if k in ('6F_gamma+iv_lag', '8F_theta+iv_lag', '8F_rho+iv_lag')}

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

## GPU auto-detect

In [3]:
CFG = detect_device()
DEVICE = CFG['DEVICE']
AMP_DTYPE = CFG['POLICY']
USE_AMP = (AMP_DTYPE != torch.float32)

L4  |  VRAM: 24 GB  |  MAX_BATCH=2,048  |  dtype=torch.bfloat16


## Train all datasets

In [4]:
RUN_DIR = make_run_dir(NOTEBOOK_NAME)
print(f'Output directory: {RUN_DIR}')

grand_total_time = 0.0

for dataset_name in DATA_SETS:
    print(f'\n{"#" * 70}')
    print(f'#  Dataset: {dataset_name}')
    print(f'{"#" * 70}')

    # Load data
    df_train = pd.read_parquet(CLEAN_DATA_V2 / f'{dataset_name}_train_v2.parquet')
    df_val   = pd.read_parquet(CLEAN_DATA_V2 / f'{dataset_name}_val_v2.parquet')
    df_test  = pd.read_parquet(CLEAN_DATA_V2 / f'{dataset_name}_test_v2.parquet')
    df_full = pd.concat([df_train, df_val, df_test], ignore_index=True)

    print(f'Train: {len(df_train):,}  Val: {len(df_val):,}  Test: {len(df_test):,}')
    print(f'Full data for sequences: {len(df_full):,} rows')

    # Analytic benchmark
    hw = analytic_benchmark(df_train, df_val, df_test, target=TARGET)
    hw_coef = hw['coef']

    # Precompute split structure (shared across all feature sets)
    print('Precomputing split structure...')
    cache = precompute_split_structure(df_full, df_train, df_val, df_test,
                                       target=TARGET, lookback=LOOKBACK)
    del df_full  # free memory; cache holds the sorted copy

    results_by_fs = {}
    df_sorted_ref = cache['df']

    pbar = tqdm(FEATURE_SETS.items(), desc=f'{dataset_name} feature sets', unit='set')
    for fs_name, feature_cols in pbar:
        pbar.set_postfix_str(fs_name)

        # Build sequences from cache (fast — no re-sort/merge)
        seq_data = build_sequences_from_cache(cache, feature_cols)
        X_tr, y_tr = seq_data['X_train'], seq_data['y_train']
        X_va, y_va = seq_data['X_val'], seq_data['y_val']
        X_te, y_te = seq_data['X_test'], seq_data['y_test']
        test_idx = seq_data['test_indices']

        print_feature_set_summary(fs_name, len(X_tr), len(X_va), len(X_te), feature_cols)

        # Scale
        X_tr_sc, X_va_sc, X_te_sc, scaler = scale_sequences(X_tr, X_va, X_te)

        # DataLoaders
        BATCH_SIZE = compute_batch_size(len(X_tr), CFG['MAX_BATCH'])
        INIT_LR = BASE_LR * (BATCH_SIZE / BASE_BATCH) ** 0.5
        print_config(CFG, BATCH_SIZE, INIT_LR, len(X_tr), MAX_EPOCHS, PATIENCE, WARMUP_EPOCHS, LOOKBACK)

        train_loader = DataLoader(SequenceDataset(X_tr_sc, y_tr), batch_size=BATCH_SIZE,
                                  shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
        val_loader = DataLoader(SequenceDataset(X_va_sc, y_va), batch_size=BATCH_SIZE,
                                shuffle=False, num_workers=2, pin_memory=True)

        # Model
        model = LSTMModel(
            n_features=len(feature_cols), hidden_size=HIDDEN_SIZE,
            num_layers=NUM_LAYERS, dropout=DROPOUT, seed=SEED,
        ).to(DEVICE)
        print(f'  LSTM params: {sum(p.numel() for p in model.parameters()):,}')

        # Train
        train_result = train_seq_model(
            model, train_loader, val_loader,
            device=DEVICE, amp_dtype=AMP_DTYPE, use_amp=USE_AMP,
            max_epochs=MAX_EPOCHS, patience=PATIENCE,
            lr_patience=LR_PATIENCE, lr_factor=LR_FACTOR,
            init_lr=INIT_LR, warmup_epochs=WARMUP_EPOCHS,
            use_tqdm=True, desc=fs_name,
        )

        # Predict & evaluate
        y_pred = predict_seq(train_result['model'], X_te_sc, DEVICE, AMP_DTYPE, USE_AMP)
        _, _, hw_sse = hw_predict_aligned(hw_coef, df_sorted_ref, test_idx)

        met = metrics(y_te, y_pred)
        g = gain(met['SSE'], hw_sse) * 100
        print(f'  {fs_name}: SSE={met["SSE"]:.4f}  RMSE={met["RMSE"]:.6f}  '
              f'Gain={g:+.2f}%  epochs={train_result["epochs"]}  '
              f'time={train_result["training_time"]:.1f}s')

        results_by_fs[fs_name] = {
            'model': train_result['model'],
            'y_pred': y_pred, 'y_true': y_te,
            'test_indices': test_idx,
            'history': train_result['history'],
            'epochs': train_result['epochs'],
            'training_time': train_result['training_time'],
            'scaler': scaler,
        }

        # Free memory
        del X_tr, X_va, X_te, X_tr_sc, X_va_sc, X_te_sc, train_loader, val_loader
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ── Save results for this dataset ─────────────────────────────────
    dataset_dir = RUN_DIR / dataset_name
    summary = save_seq_run(
        dataset_dir,
        results_by_fs=results_by_fs,
        hw_coef=hw_coef,
        df_sorted=df_sorted_ref,
    )
    print(f'\nMetrics Summary ({dataset_name}):')
    display(summary)

    # ── Dataset summary ───────────────────────────────────────────────
    ds_time = sum(r['training_time'] for r in results_by_fs.values())
    grand_total_time += ds_time
    print(f'\nLSTM on {dataset_name} — training time: {ds_time / 60:.1f} min')
    for fs_name, res in results_by_fs.items():
        met = metrics(res['y_true'], res['y_pred'])
        _, _, hw_sse = hw_predict_aligned(hw_coef, df_sorted_ref, res['test_indices'])
        g = gain(met['SSE'], hw_sse) * 100
        print(f'  {fs_name}: SSE={met["SSE"]:.4f}  Gain={g:+.2f}%  epochs={res["epochs"]}')

    # Free memory before next dataset
    del results_by_fs, df_sorted_ref, df_train, df_val, df_test, cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Output directory: /content/drive/MyDrive/Colab Notebooks/IV Project-Apr 3, 2026/output/3.1-lstm-chro-B-C-D/01-run

######################################################################
#  Dataset: chro_B
######################################################################
Train: 744,477  Val: 331,626  Test: 225,573
Full data for sequences: 1,301,676 rows
Analytic Benchmark
SSE = 39.5613  RMSE = 0.013243
Coefficients: a = -0.082956, b = -0.181799, c = -0.174770
Precomputing split structure...


chro_B feature sets:   0%|          | 0/3 [00:00<?, ?set/s]


  Feature set: 6F_gamma+iv_lag  (7 features)
  Train: 483,197  Val: 196,442  Test: 145,296 sequences
  Features: delta, T, spy_ret, vix_lag, vix_mom_lag, gamma, iv_lag
MAX_BATCH=2,048  adaptive BATCH_SIZE=2,048  INIT_LR=0.002000  n_train=483,197  steps/epoch~235
MAX_EPOCHS=100  PATIENCE=12  WARMUP=5 epochs  LOOKBACK=20
  LSTM params: 52,033


6F_gamma+iv_lag:   0%|          | 0/100 epochs [00:00<?]  

  6F_gamma+iv_lag: SSE=1.7923  RMSE=0.003512  Gain=+91.73%  epochs=94  time=360.7s

  Feature set: 8F_theta+iv_lag  (9 features)
  Train: 483,197  Val: 196,442  Test: 145,296 sequences
  Features: delta, T, spy_ret, vix_lag, vix_mom_lag, vix_mom, gamma, theta, iv_lag
MAX_BATCH=2,048  adaptive BATCH_SIZE=2,048  INIT_LR=0.002000  n_train=483,197  steps/epoch~235
MAX_EPOCHS=100  PATIENCE=12  WARMUP=5 epochs  LOOKBACK=20
  LSTM params: 52,545


...

  8F_theta+iv_lag: SSE=2.1050  RMSE=0.004444  Gain=+57.96%  epochs=17  time=50.2s

  Feature set: 8F_rho+iv_lag  (9 features)
  Train: 561,633  Val: 195,325  Test: 106,564 sequences
  Features: delta, T, spy_ret, vix_lag, vix_mom_lag, vix_mom, gamma, rho, iv_lag
MAX_BATCH=2,048  adaptive BATCH_SIZE=2,048  INIT_LR=0.002000  n_train=561,633  steps/epoch~274
MAX_EPOCHS=100  PATIENCE=12  WARMUP=5 epochs  LOOKBACK=20
  LSTM params: 52,545


8F_rho+iv_lag:   0%|          | 0/100 epochs [00:00<?]  

  8F_rho+iv_lag: SSE=3.8249  RMSE=0.005991  Gain=+23.61%  epochs=14  time=40.9s
Saving results to: /content/drive/MyDrive/Colab Notebooks/IV Project-Apr 3, 2026/output/3.1-lstm-chro-B-C-D/01-run/chro_D
Feature sets to save: ['6F_gamma+iv_lag', '8F_theta+iv_lag', '8F_rho+iv_lag']

✓ Saved metrics_summary.csv
✓ Saved residual_diagnostics.csv
✓ Saved gain_table.csv
✓ Saved 3 × 3 artifacts (weights, predictions, history)
✓ All results saved to /content/drive/MyDrive/Colab Notebooks/IV Project-Apr 3, 2026/output/3.1-lstm-chro-B-C-D/01-run/chro_D

Metrics Summary (chro_D):


,Model,SSE,MSE,RMSE,MAE,MeanError,MedianAE,R2,Training_time,Gain_vs_Analytic,Gain_Incremental
0,Analytic,5.007344,0.000047,0.006855,0.003177,0.000577,0.001452,0.072457,None,None,None
1,6F_gamma+iv_lag,4.602337,0.000043,0.006572,0.003571,0.000915,0.002197,0.147479,54.6s,8.09%,None
2,8F_theta+iv_lag,2.104970,0.000020,0.004444,0.002477,-0.000530,0.001439,0.610083,50.2s,57.96%,54.26%
3,8F_rho+iv_lag,3.824899,0.000036,0.005991,0.002913,0.000703,0.001616,0.291489,40.9s,23.61%,-81.71%
4,Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,145.6s,None,None



LSTM on chro_D — training time: 2.4 min
  6F_gamma+iv_lag: SSE=4.6023  Gain=+8.09%  epochs=13
  8F_theta+iv_lag: SSE=2.1050  Gain=+57.96%  epochs=17
  8F_rho+iv_lag: SSE=3.8249  Gain=+23.61%  epochs=14


## Grand Summary

In [5]:
print(f'\n{"=" * 70}')
print(f'LSTM on all datasets — grand total training time: {grand_total_time / 60:.1f} min')
print(f'Results saved to: {RUN_DIR}')
for ds in DATA_SETS:
    print(f'  {ds}: {RUN_DIR / ds}')


LSTM on all datasets — grand total training time: 27.5 min
Results saved to: /content/drive/MyDrive/Colab Notebooks/IV Project-Apr 3, 2026/output/3.1-lstm-chro-B-C-D/01-run
  chro_B: /content/drive/MyDrive/Colab Notebooks/IV Project-Apr 3, 2026/output/3.1-lstm-chro-B-C-D/01-run/chro_B
  chro_C: /content/drive/MyDrive/Colab Notebooks/IV Project-Apr 3, 2026/output/3.1-lstm-chro-B-C-D/01-run/chro_C
  chro_D: /content/drive/MyDrive/Colab Notebooks/IV Project-Apr 3, 2026/output/3.1-lstm-chro-B-C-D/01-run/chro_D


## Cleanup

In [6]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'Final VRAM: {(total - free) / 1e9:.2f} GB / {total / 1e9:.0f} GB')

print(f'Grand total training time: {grand_total_time / 60:.1f} min')

if IN_COLAB:
    runtime.unassign()

Final VRAM: 0.42 GB / 24 GB
Grand total training time: 27.5 min
